# Basic Analysis — Questions 1-10

**Techniques:** `GROUP BY` · `JOIN` · `DISTINCT` · `LEFT JOIN + NULL` · `HAVING` · correlated subquery · `UNION ALL`

| # | Question |
|---|----------|
| Q1 | Top product categories by order volume |
| Q2 | Distribution of order statuses |
| Q3 | States with the most customers |
| Q4 | Average order value by payment type |
| Q5 | Average review score by product category |
| Q6 | How many customers are repeat buyers? |
| Q7 | What percentage of orders are missing a review? |
| Q8 | Categories with both high volume AND high satisfaction |
| Q9 | Products priced above their category average |
| Q10 | States that appear as both customer and seller locations |

In [ ]:
import sys
sys.path.append('../')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.db_utils import run_query
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## Q1 — Top 15 Product Categories by Order Volume
`GROUP BY`, `ORDER BY`

**Why?** Understanding which categories drive the most orders is the starting point for any e-commerce analysis. It tells us where inventory, logistics, and marketing effort is concentrated — and where the business depends most on things going right.

> **Key insight:** Bed & bath leads with 11k+ orders, but don't confuse volume with value — electronics and computers have fewer orders yet much higher average ticket prices. Volume rankings and revenue rankings tell very different stories.

In [ ]:
q1 = run_query("""
    SELECT t.product_category_name_english AS category,
           COUNT(oi.order_id) AS total_orders
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN product_category_name_translation t
        ON p.product_category_name = t.product_category_name
    GROUP BY category ORDER BY total_orders DESC LIMIT 15
""")
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=q1, x='total_orders', y='category', ax=ax, color='steelblue')
ax.set_title('Q1 — Top 15 Product Categories by Order Volume', fontweight='bold')
ax.set_xlabel('Total Orders'); ax.set_ylabel('')
plt.tight_layout(); plt.savefig('../images/q1_top_categories.png', bbox_inches='tight'); plt.show()
q1

## Q2 — Order Status Distribution
`GROUP BY`, window `COUNT`

**Why?** A clean status breakdown catches data quality issues early and sets the baseline for every filtered query that follows. Most analyses only make sense on 'delivered' orders — knowing exactly how many that is matters.

> **Key insight:** 97% of orders reach "delivered" status — Olist's logistics backbone holds up well. The 0.6% cancellation rate skews toward boleto (bank slip) orders that expire before fulfilment — a known payment friction in Brazilian e-commerce.

In [ ]:
q2 = run_query("""
    SELECT order_status, COUNT(*) AS total_orders,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
    FROM orders GROUP BY order_status ORDER BY total_orders DESC
""")
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(q2['total_orders'], labels=q2['order_status'], autopct='%1.1f%%', startangle=140)
ax.set_title('Q2 — Order Status Distribution', fontweight='bold')
plt.tight_layout(); plt.savefig('../images/q2_order_status.png', bbox_inches='tight'); plt.show()
q2

## Q3 — Top 10 States by Customer Count
`COUNT DISTINCT`, `GROUP BY`

**Why?** Brazil is geographically massive and economically uneven. Understanding where customers are concentrated shapes logistics planning, regional marketing spend, and delivery time expectations.

> **Key insight:** São Paulo alone accounts for ~42% of all customers (41,746 of ~100k). The top 3 states (SP, RJ, MG) together cover ~66% of demand. Any logistics or marketing decision should weight SP heavily — it's not just the biggest market, it's almost half the market.

In [ ]:
q3 = run_query("""
    SELECT customer_state AS state,
           COUNT(DISTINCT customer_id) AS unique_customers
    FROM customers GROUP BY state ORDER BY unique_customers DESC LIMIT 10
""")
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=q3, x='state', y='unique_customers', ax=ax, color='coral')
ax.set_title('Q3 — Top 10 States by Customer Count', fontweight='bold')
plt.tight_layout(); plt.savefig('../images/q3_customers_by_state.png', bbox_inches='tight'); plt.show()
q3

## Q4 — Average Order Value by Payment Type
`AVG`, `SUM`, `GROUP BY`

**Why?** Payment method is a proxy for purchase intent and financial behaviour. Credit card installments (parcelamento) enable larger purchases in Brazil — this query tests that hypothesis with data.

> **Key insight:** Credit card orders average R$163 — 2.5× higher than voucher orders (R$66). Brazilian credit cards support installments (parcelamento), letting customers split large purchases across up to 12 monthly payments. Vouchers are capped at face value.

In [ ]:
q4 = run_query("""
    SELECT payment_type, COUNT(DISTINCT order_id) AS total_orders,
           ROUND(AVG(payment_value), 2) AS avg_order_value,
           ROUND(SUM(payment_value), 2) AS total_revenue
    FROM order_payments GROUP BY payment_type ORDER BY total_revenue DESC
""")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=q4, x='payment_type', y='avg_order_value', ax=axes[0], color='mediumseagreen')
axes[0].set_title('Avg Order Value'); axes[0].set_xlabel('')
sns.barplot(data=q4, x='payment_type', y='total_revenue', ax=axes[1], color='mediumpurple')
axes[1].set_title('Total Revenue'); axes[1].set_xlabel('')
plt.suptitle('Q4 — Payment Type Analysis', fontweight='bold')
plt.tight_layout(); plt.savefig('../images/q4_payment_analysis.png', bbox_inches='tight'); plt.show()
q4

## Q5 — Avg Review Score by Category (min 50 reviews)
Multi-table `JOIN`, `HAVING`

**Why?** Category-level satisfaction reveals quality and reliability gaps that aggregate averages hide. The minimum review filter prevents small-sample noise — a category with 2 reviews at 5.0 stars is not meaningful evidence.

> **Key insight:** Books and construction tools top the satisfaction charts. Categories where the product is exactly what it appears to be (commodity items) score higher. Categories with sizing, fit, or quality variance score lower — expectation mismatch is the main driver of bad reviews.

In [ ]:
q5 = run_query("""
    SELECT t.product_category_name_english AS category,
           COUNT(r.review_id) AS total_reviews,
           ROUND(AVG(r.review_score), 2) AS avg_score
    FROM order_reviews r
    JOIN orders o       ON r.order_id = o.order_id
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p     ON oi.product_id = p.product_id
    JOIN product_category_name_translation t
        ON p.product_category_name = t.product_category_name
    GROUP BY category HAVING total_reviews > 50
    ORDER BY avg_score DESC LIMIT 15
""")
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if s >= 4.0 else '#e74c3c' for s in q5['avg_score']]
sns.barplot(data=q5, x='avg_score', y='category', ax=ax, palette=colors, hue=q5['avg_score'], legend=False)
ax.axvline(x=4.0, color='gray', linestyle='--', linewidth=1)
ax.set_title('Q5 — Top 15 Categories by Avg Review Score', fontweight='bold')
ax.set_xlabel('Average Score (1-5)'); ax.set_ylabel('')
plt.tight_layout(); plt.savefig('../images/q5_review_by_category.png', bbox_inches='tight'); plt.show()
q5

## Q6 — How Many Customers Are Repeat Buyers?
`DISTINCT` + scalar subquery + `HAVING`

**Why?** Customer retention is more profitable than acquisition. This measures the repeat purchase rate — a core e-commerce health metric. If nearly all customers only buy once, that is a structural problem no logistics optimisation can fix.

> **Key insight:** Only ~3% of customers place more than one order. Olist's marketplace structure means customers often buy once and move on to competitor platforms. Repeat purchase rate is the single most important growth lever the business hasn't cracked yet.

In [ ]:
q6_repeat = run_query("""
    SELECT COUNT(DISTINCT customer_unique_id) AS repeat_customers
    FROM (
        SELECT c.customer_unique_id, COUNT(o.order_id) AS order_count
        FROM customers c JOIN orders o ON c.customer_id = o.customer_id
        GROUP BY c.customer_unique_id HAVING order_count > 1
    ) sub
""")
total_cust = run_query("SELECT COUNT(DISTINCT customer_unique_id) AS total FROM customers")
repeat = q6_repeat['repeat_customers'].values[0]
tot    = total_cust['total'].values[0]
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie([repeat, tot - repeat], labels=['Repeat Buyers', 'One-time Buyers'],
       autopct='%1.1f%%', colors=['#3498db', '#ecf0f1'], startangle=90)
ax.set_title(f'Q6 — Repeat vs One-time Buyers\n({tot:,} total customers)', fontweight='bold')
plt.tight_layout(); plt.savefig('../images/q6_repeat_buyers.png', bbox_inches='tight'); plt.show()
print(f'Repeat buyers: {repeat:,} ({repeat/tot*100:.1f}%)')

## Q7 — Missing Reviews: What % of Orders Have No Review?
`LEFT JOIN`, `NULL` check

**Why?** Before using review scores in any analysis, we need to know how complete that data is. If a large share of orders have no review, any review-based metric is a biased sample.

> **Key insight:** Only 0.8% of orders are missing a review — Olist's automated review-request emails are highly effective. The catch: a near-mandatory review system can inflate scores if customers feel pressured to respond positively.

In [ ]:
q7 = run_query("""
    SELECT COUNT(*) AS total_orders,
           SUM(CASE WHEN r.review_id IS NULL THEN 1 ELSE 0 END) AS missing_reviews,
           ROUND(SUM(CASE WHEN r.review_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS missing_pct
    FROM orders o
    LEFT JOIN order_reviews r ON o.order_id = r.order_id
""")
missing = q7['missing_reviews'].values[0]
has_rev = q7['total_orders'].values[0] - missing
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie([has_rev, missing], labels=['Has Review', 'Missing Review'],
       autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
ax.set_title('Q7 — Orders Missing a Review', fontweight='bold')
plt.tight_layout(); plt.savefig('../images/q7_missing_reviews.png', bbox_inches='tight'); plt.show()
q7

## Q8 — High Volume AND High Satisfaction Categories
`HAVING` with multiple conditions

**Why?** Volume and satisfaction pulled separately can mislead. Combining both filters finds categories that are commercially important and delivering good experiences — the business's real strengths.

> **Key insight:** Very few categories sit in the top quartile for both volume *and* satisfaction — this intersection is the sweet spot for marketplace expansion. High volume + low satisfaction flags product quality or logistics problems worth fixing.

In [ ]:
q8 = run_query("""
    SELECT t.product_category_name_english AS category,
           COUNT(oi.order_id) AS total_orders,
           ROUND(AVG(r.review_score), 2) AS avg_score
    FROM order_items oi
    JOIN products p     ON oi.product_id = p.product_id
    JOIN product_category_name_translation t ON p.product_category_name = t.product_category_name
    JOIN orders o       ON oi.order_id = o.order_id
    JOIN order_reviews r ON o.order_id = r.order_id
    GROUP BY category
    HAVING total_orders > 500 AND avg_score > 4.0
    ORDER BY avg_score DESC
""")
fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(q8['total_orders'], q8['avg_score'],
                     s=120, c=q8['avg_score'], cmap='RdYlGn', vmin=3.8, vmax=5.0, zorder=3)
for _, row in q8.iterrows():
    ax.annotate(row['category'], (row['total_orders'], row['avg_score']),
                fontsize=7, ha='left', va='bottom')
ax.axhline(4.0, color='gray', linestyle='--', linewidth=1)
ax.axvline(500, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Total Orders'); ax.set_ylabel('Avg Review Score')
ax.set_title('Q8 — Categories with High Volume AND High Satisfaction', fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Avg Score')
plt.tight_layout(); plt.savefig('../images/q8_high_vol_high_sat.png', bbox_inches='tight'); plt.show()
print(f'{len(q8)} categories meet both criteria (>500 orders AND >4.0 avg score)')
q8

## Q9 — Products Priced Above Their Category Average
Correlated subquery

**Why?** This introduces a key SQL pattern: comparing each row to its group's aggregate using a subquery. The technique (filtering with a correlated subquery) is the transferable skill here.

> **Key insight:** Price outliers within a category often indicate premium or bundle listings. A product priced 2× the category average isn't necessarily overpriced — it may serve a niche. But if it also carries low review scores, it's a candidate for seller education or listing cleanup.

In [ ]:
q9 = run_query("""
    SELECT oi.product_id,
           t.product_category_name_english AS category,
           ROUND(AVG(oi.price), 2) AS product_avg_price,
           ROUND((
               SELECT AVG(oi2.price) FROM order_items oi2
               JOIN products p2 ON oi2.product_id = p2.product_id
               WHERE p2.product_category_name = p.product_category_name
           ), 2) AS category_avg_price
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN product_category_name_translation t ON p.product_category_name = t.product_category_name
    GROUP BY oi.product_id, category
    HAVING product_avg_price > category_avg_price
    ORDER BY category, product_avg_price DESC LIMIT 20
""")
q9['price_premium'] = q9['product_avg_price'] - q9['category_avg_price']
fig, ax = plt.subplots(figsize=(10, 6))
top15 = q9.nlargest(15, 'price_premium')
sns.barplot(data=top15, x='price_premium', y='category', ax=ax, color='#e67e22')
ax.set_title('Q9 — Price Premium Above Category Average (Top 15)', fontweight='bold')
ax.set_xlabel('Price Premium (BRL)'); ax.set_ylabel('')
plt.tight_layout(); plt.savefig('../images/q9_price_premium.png', bbox_inches='tight'); plt.show()
q9.head(10)

## Q10 — States: Customer vs Seller Geographic Overlap
`UNION ALL`, `IN` subquery

**Why?** Most Brazilian orders cross state lines — sellers concentrate in SP while customers spread nationwide. This query quantifies that mismatch, which directly explains delivery times and freight costs in later analyses.

> **Key insight:** SP, RJ, and MG appear in both customer and seller rankings. The same states that generate demand also supply most of the inventory. This geographic concentration creates logistics efficiency but also fragility: disruptions in SP ripple across both supply and demand simultaneously.

In [ ]:
q10 = run_query("""
    SELECT customer_state AS state, 'Customer + Seller' AS presence
    FROM customers
    WHERE customer_state IN (SELECT seller_state FROM sellers)
    GROUP BY customer_state
    UNION ALL
    SELECT customer_state AS state, 'Customer only' AS presence
    FROM customers
    WHERE customer_state NOT IN (SELECT seller_state FROM sellers)
    GROUP BY customer_state
    ORDER BY presence, state
""")
counts = q10['presence'].value_counts()
both = q10[q10['presence'] == 'Customer + Seller']['state'].tolist()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(counts.index, counts.values,
            color=['#3498db' if 'Both' in p else '#bdc3c7' for p in counts.index])
axes[0].set_title('State Presence Count'); axes[0].set_ylabel('Number of States')
axes[1].axis('off')
axes[1].text(0.05, 0.95,
    'States with BOTH Customer & Seller presence:\n\n' + ', '.join(sorted(both)),
    transform=axes[1].transAxes, fontsize=9, va='top', family='monospace')
plt.suptitle('Q10 — Geographic Overlap: Customers vs Sellers', fontweight='bold')
plt.tight_layout(); plt.savefig('../images/q10_state_overlap.png', bbox_inches='tight'); plt.show()
print(f'States with seller presence: {len(both)} / {q10["state"].nunique()}')